# Qwen3 0.6B: GRPO training

Open this notebook locally in VS Code and select your Nebius GPU kernel, just like the other chapter notebooks. No repository path or clone is required: the training script is embedded below and imports the installed `reasoning_from_scratch` package.

The model and dataset download automatically when needed. Outputs are written in the active kernel's working folder. With a remote kernel, download checkpoints to your computer before removing the remote storage.

Configuration: 100 steps, eight rollouts, up to 1,024 generated tokens, evaluation on 50 problems every 50 steps. This is real full-weight fine-tuning, not a full dataset pass. Published results used a 512-token training limit.

**After CUDA out-of-memory: use the Clear GPU memory cell below.** Restart the kernel if allocated memory remains high. This version processes rollout gradients sequentially; if a single 1,024-token rollout still exceeds memory, set MAX_NEW_TOKENS to 512.


## Dependencies
Uses the same package installation source as your chapter 6 and 7 notebooks. Skip installation if the package is already installed. Restart the kernel if installation updates libraries that were already imported.

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("reasoning_from_scratch") is None:
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "git+https://github.com/RRisto/reasoning-from-scratch.git"
    ], check=True)
else:
    print("reasoning_from_scratch is already installed.")


reasoning_from_scratch is already installed.


In [2]:
from pathlib import Path
import torch

assert torch.cuda.is_available(), "Select the Nebius GPU kernel with CUDA-enabled PyTorch."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
OUTPUT_DIR = (Path.cwd() / "grpo_training").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Training outputs:", OUTPUT_DIR)


GPU: NVIDIA L40S
GPU memory (GiB): 44.4
Training outputs: /home/jovyan/grpo_training


## Training settings
Set `STEPS = None` for one pass through the dataset. Longer training does not guarantee higher accuracy. Keep both checkpoints. Rerunning in the same output folder may overwrite matching checkpoint names.

In [3]:
STEPS = 100
NUM_ROLLOUTS = 8
MAX_NEW_TOKENS = 1024
EVAL_EXAMPLES = 50


## Embedded training script
This version accumulates gradients one rollout at a time to reduce peak GPU memory. It keeps the original mean GRPO objective and one optimizer update per group. Zero-advantage groups are skipped.


In [4]:
TRAINING_SOURCE = '# Copyright (c) Sebastian Raschka under Apache License 2.0 (see LICENSE.txt)\n# Source for "Build a Reasoning Model (From Scratch)": https://mng.bz/lZ5B\n# Code repository: https://github.com/rasbt/reasoning-from-scratch\n\nimport argparse\nimport time\nfrom pathlib import Path\n\nimport torch\n\nfrom reasoning_from_scratch.ch02 import get_device\nfrom reasoning_from_scratch.ch03 import (\n    evaluate_math500_stream,\n    render_prompt,\n    extract_final_candidate,\n    grade_answer,\n    eta_progress_message,\n    load_model_and_tokenizer,\n    load_math500_test,\n    load_tokenizer_only,\n)\nfrom reasoning_from_scratch.ch04 import top_p_filter\nfrom reasoning_from_scratch.ch06 import (\n    load_math_train,\n)\nfrom reasoning_from_scratch.qwen3 import KVCache, Qwen3Model, QWEN_CONFIG_06_B\n\nSCRIPT_NAME = Path(__file__).stem\nLOG_PATH = Path(__file__).parent / "logs" / f"{SCRIPT_NAME}_outputs.txt"\nMETRICS_LOG_PATH = Path(__file__).parent / "logs" / f"{SCRIPT_NAME}_metrics.txt"\nCSV_LOG_PATH = Path(__file__).parent / "logs" / f"{SCRIPT_NAME}_metrics.csv"\nCHECKPOINT_DIR = Path(__file__).parent / "checkpoints" / SCRIPT_NAME\n\n\n@torch.no_grad()\ndef sample_response(\n    model,\n    tokenizer,\n    prompt,\n    device,\n    max_new_tokens=512,\n    temperature=0.8,\n    top_p=0.9,\n):\n    input_ids = torch.tensor(\n        tokenizer.encode(prompt),\n        device=device\n        )\n\n    cache = KVCache(n_layers=model.cfg["n_layers"])\n    model.reset_kv_cache()\n    logits = model(input_ids.unsqueeze(0), cache=cache)[:, -1]\n\n    generated = []\n    for step in range(max_new_tokens):\n        if temperature and temperature != 1.0:\n            logits = logits / temperature\n\n        probas = torch.softmax(logits, dim=-1)\n        probas = top_p_filter(probas, top_p)\n\n        # In the core chapters, we used .cpu() for better consistency across systems,\n        # but it causes a 20% performance hit when training on GPUs\n        # next_token = torch.multinomial(probas.cpu(), num_samples=1).to(device)\n        next_token = torch.multinomial(probas, num_samples=1)\n\n        token_id = next_token.item()\n        generated.append(token_id)\n\n        if (\n            tokenizer.eos_token_id is not None\n            and token_id == tokenizer.eos_token_id\n        ):\n            break\n        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated\n            logits = model(next_token, cache=cache)[:, -1]\n\n    full_token_ids = torch.cat(\n        [input_ids,\n         torch.tensor(generated, device=device, dtype=input_ids.dtype),]\n    )\n    return full_token_ids, input_ids.numel(), tokenizer.decode(generated)\n\n\ndef sequence_logprob(model, token_ids, prompt_len):\n    logits = model(token_ids.unsqueeze(0)).squeeze(0).float()\n    logprobs = torch.log_softmax(logits, dim=-1)\n\n    targets = token_ids[1:]\n    selected = logprobs[:-1].gather(1, targets.unsqueeze(-1)).squeeze(-1)\n    return selected[prompt_len - 1:].sum()\n\n\ndef reward_rlvr(answer_text, ground_truth):\n    extracted = extract_final_candidate(\n        answer_text, fallback=None  # Require \\boxed{}\n    )\n    if not extracted:\n        return 0.0\n    correct = grade_answer(extracted, ground_truth)\n    return float(correct)\n\n\ndef compute_grpo_loss(\n    model,\n    tokenizer,\n    example,\n    device,\n    num_rollouts=4,\n    max_new_tokens=512,\n    temperature=0.8,\n    top_p=0.9,\n    skip_zero_adv=False,\n):\n    roll_rewards, samples, rollout_data = [], [], []\n    prompt = render_prompt(example["problem"])\n\n    was_training = model.training\n    model.eval()\n\n    for _ in range(num_rollouts):\n        token_ids, prompt_len, text = sample_response(\n            model=model,\n            tokenizer=tokenizer,\n            prompt=prompt,\n            device=device,\n            max_new_tokens=max_new_tokens,\n            temperature=temperature,\n            top_p=top_p,\n        )\n        reward = reward_rlvr(text, example["answer"])\n\n        roll_rewards.append(reward)\n        rollout_data.append((token_ids, prompt_len))\n        samples.append(\n            {\n                "text": text,\n                "reward": reward,\n                "gen_len": token_ids.numel() - prompt_len,\n            }\n        )\n\n    if was_training:\n        model.train()\n\n    rewards = torch.tensor(roll_rewards, device=device)\n    advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)\n\n    is_zero_adv = torch.allclose(\n        advantages,\n        torch.zeros_like(advantages),\n        atol=1e-8,\n        rtol=0.0,\n    )\n\n    if skip_zero_adv and is_zero_adv:\n        return {\n            "loss": 0.0,\n            "pg_loss": 0.0,\n            "rewards": roll_rewards,\n            "advantages": advantages.detach().cpu().tolist(),\n            "is_zero_adv": True,\n            "samples": samples,\n            "loss_tensor": None,\n        }\n\n    # Accumulate the same mean policy gradient without retaining every\n    # rollout\'s activation graph until the end of the group.\n    loss = torch.zeros((), device=device)\n    for advantage, (token_ids, prompt_len) in zip(advantages.detach(), rollout_data):\n        if advantage.item() == 0.0:\n            continue\n        logp = sequence_logprob(model, token_ids, prompt_len)\n        rollout_loss = -advantage * logp / len(rollout_data)\n        rollout_loss.backward()\n        loss += rollout_loss.detach()\n        del logp, rollout_loss\n    if is_zero_adv:\n        for parameter in model.parameters():\n            if parameter.requires_grad:\n                parameter.grad = torch.zeros_like(parameter)\n    pg_loss = loss\n\n    return {\n        "loss": loss.item(),\n        "pg_loss": pg_loss.item(),\n        "rewards": roll_rewards,\n        "advantages": advantages.detach().cpu().tolist(),\n        "is_zero_adv": is_zero_adv,\n        "samples": samples,\n        "loss_tensor": loss,\n    }\n\n\ndef append_sample_logs(step_idx, samples, max_samples=3):\n    LOG_PATH.parent.mkdir(parents=True, exist_ok=True)\n    with LOG_PATH.open("a", encoding="utf-8") as f:\n        f.write(f"[Step {step_idx}] sample outputs\\n")\n        for i, sample in enumerate(samples[:max_samples]):\n            text = sample["text"].replace("\\n", "\\\\n")\n            f.write(\n                f"  {i+1}) reward={sample[\'reward\']:.3f} "\n                f"len={sample[\'gen_len\']}: {text}\\n"\n            )\n        f.write("\\n")\n\n\ndef append_step_metrics(\n    step_idx,\n    total_steps,\n    loss,\n    reward_avg,\n    tokens_per_sec,\n    avg_response_len,\n    eval_acc=None,\n):\n    METRICS_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)\n    with METRICS_LOG_PATH.open("a", encoding="utf-8") as f:\n        f.write(\n            f"[Step {step_idx}/{total_steps}] "\n            f"loss={loss:.4f} reward_avg={reward_avg:.3f} "\n            f"tokens_per_sec={tokens_per_sec:.1f} "\n            f"avg_response_len={avg_response_len:.1f}\\n"\n        )\n    CSV_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)\n    if not CSV_LOG_PATH.exists():\n        CSV_LOG_PATH.write_text(\n            "step,total_steps,loss,reward_avg,tokens_per_sec,avg_response_len,eval_acc\\n",\n            encoding="utf-8",\n        )\n    with CSV_LOG_PATH.open("a", encoding="utf-8") as f:\n        eval_acc_str = "" if eval_acc is None else f"{eval_acc:.6f}"\n        f.write(\n            f"{step_idx},{total_steps},{loss:.6f},{reward_avg:.6f},"\n            f"{tokens_per_sec:.6f},{avg_response_len:.6f},{eval_acc_str}\\n"\n        )\n\n\ndef append_eval_metrics(step_idx, acc, correct, total):\n    METRICS_LOG_PATH.parent.mkdir(parents=True, exist_ok=True)\n    with METRICS_LOG_PATH.open("a", encoding="utf-8") as f:\n        f.write(\n            f"[Eval step {step_idx}] math500_acc={acc:.4f} "\n            f"({correct}/{total})\\n"\n        )\n\n\ndef save_checkpoint(model, checkpoint_dir, step, suffix=""):\n    checkpoint_dir = Path(checkpoint_dir)\n    checkpoint_dir.mkdir(parents=True, exist_ok=True)\n    suffix = f"-{suffix}" if suffix else ""\n    ckpt_path = checkpoint_dir / f"qwen3-0.6B-rlvr-grpo-step{step:05d}{suffix}.pth"\n    torch.save(model.state_dict(), ckpt_path)\n    return ckpt_path\n\n\ndef train_rlvr_grpo(\n    model,\n    tokenizer,\n    math_data,\n    math500_eval_data,\n    device,\n    steps=None,\n    num_rollouts=9,\n    max_new_tokens=512,\n    temperature=0.8,\n    top_p=0.9,\n    lr=1e-5,\n    checkpoint_every=50,\n    checkpoint_dir=CHECKPOINT_DIR,\n    eval_max_items=0,\n    skip_zero_advantage_updates=False,\n    show_eta=False,\n):\n    if steps is None:\n        steps = len(math_data)\n\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)\n    model.train()\n    current_step = 0\n    train_start_time = time.time() if show_eta else None\n    try:\n        for step in range(steps):\n            step_start = time.perf_counter()\n            current_step = step + 1\n            example = math_data[step % len(math_data)]\n            optimizer.zero_grad(set_to_none=True)\n            stats = compute_grpo_loss(\n                model=model,\n                tokenizer=tokenizer,\n                example=example,\n                device=device,\n                num_rollouts=num_rollouts,\n                max_new_tokens=max_new_tokens,\n                temperature=temperature,\n                top_p=top_p,\n                skip_zero_adv=skip_zero_advantage_updates,\n            )\n            if stats["loss_tensor"] is not None:\n                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n                optimizer.step()\n\n            reward_avg = torch.tensor(stats["rewards"]).mean().item()\n            step_time = time.perf_counter() - step_start\n            step_tokens = sum(sample["gen_len"] for sample in stats["samples"])\n            avg_response_len = (\n                step_tokens / len(stats["samples"]) if stats["samples"] else 0.0\n            )\n            tokens_per_sec = step_tokens / step_time if step_time > 0 else 0.0\n            if current_step % 10 == 0:\n                append_sample_logs(current_step, stats["samples"])\n\n            eval_acc = None\n            if checkpoint_every and current_step % checkpoint_every == 0:\n                ckpt_path = save_checkpoint(\n                    model=model,\n                    checkpoint_dir=checkpoint_dir,\n                    step=current_step,\n                )\n                print(f"Saved checkpoint to {ckpt_path}")\n                if eval_max_items and math500_eval_data:\n                    was_training = model.training\n                    model.eval()\n                    subset = (\n                        math500_eval_data[:eval_max_items]\n                        if eval_max_items\n                        else math500_eval_data\n                    )\n                    out_path = (\n                        Path(checkpoint_dir)\n                        / f"{SCRIPT_NAME}-step{current_step:05d}-math500.jsonl"\n                    )\n                    num_correct, num_examples, acc = evaluate_math500_stream(\n                        model=model,\n                        tokenizer=tokenizer,\n                        device=device,\n                        math_data=subset,\n                        out_path=out_path,\n                        max_new_tokens=max_new_tokens,\n                        verbose=False,\n                    )\n                    eval_acc = acc\n                    append_eval_metrics(current_step, acc, num_correct, num_examples)\n                    print(\n                        f"MATH-500 eval @ step {current_step}: "\n                        f"acc={acc:.3f} ({num_correct}/{num_examples})"\n                    )\n                    if was_training:\n                        model.train()\n\n            append_step_metrics(\n                current_step,\n                steps,\n                stats["loss"],\n                reward_avg,\n                tokens_per_sec,\n                avg_response_len,\n                eval_acc=eval_acc,\n            )\n\n            eta_suffix = ""\n            if show_eta:\n                eta_msg = eta_progress_message(\n                    processed=current_step,\n                    total=steps,\n                    start_time=train_start_time,\n                    show_eta=True,\n                    label="Step",\n                ).rstrip()\n                eta_part = eta_msg.split(" | ", 1)[-1]\n                eta_suffix = f" | {eta_part}"\n            print(\n                f"[Step {current_step}/{steps}] "\n                f"loss={stats[\'loss\']:.4f} "\n                f"reward_avg={reward_avg:.3f} "\n                f"tok/sec={tokens_per_sec:.1f} "\n                f"avg_resp_len={avg_response_len:.1f}"\n                f"{eta_suffix}"\n            )\n    except KeyboardInterrupt:\n        ckpt_path = save_checkpoint(\n            model=model,\n            checkpoint_dir=checkpoint_dir,\n            step=max(1, current_step),\n            suffix="interrupt",\n        )\n        print(f"\\nKeyboardInterrupt. Saved checkpoint to {ckpt_path}")\n        return model\n    return model\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(\n        formatter_class=argparse.ArgumentDefaultsHelpFormatter,\n        description="Train RLVR GRPO on the MATH dataset."\n    )\n    parser.add_argument(\n        "--steps",\n        type=int,\n        default=None,\n        help="Number of training steps.",\n    )\n    parser.add_argument(\n        "--num_rollouts",\n        type=int,\n        default=8,\n        help="Number of rollouts per step.",\n    )\n    parser.add_argument(\n        "--max_new_tokens",\n        type=int,\n        default=512,\n        help="Maximum tokens to generate per rollout.",\n    )\n    parser.add_argument(\n        "--temperature",\n        type=float,\n        default=0.8,\n        help="Sampling temperature.",\n    )\n    parser.add_argument(\n        "--top_p",\n        type=float,\n        default=0.9,\n        help="Top-p sampling cutoff.",\n    )\n    parser.add_argument(\n        "--seed",\n        type=str,\n        default="42",\n        help="Random seed (int) or None to disable seeding.",\n    )\n    parser.add_argument(\n        "--checkpoint_path",\n        type=str,\n        default=None,\n        help="Optional path to a .pth checkpoint to resume training from.",\n    )\n    parser.add_argument(\n        "--eval_on_checkpoint",\n        type=int,\n        default=0,\n        help=(\n            "Number of MATH-500 examples to evaluate at checkpoints "\n            "(0 disables)."\n        ),\n    )\n    parser.add_argument(\n        "--skip-zero-advantage-updates",\n        action="store_true",\n        help=(\n            "Skip backward/optimizer step when rollout advantages are all "\n            "near zero."\n        ),\n    )\n    parser.add_argument(\n        "--show_eta",\n        action="store_true",\n        help="Append ETA to step logs.",\n    )\n    args = parser.parse_args()\n\n    if args.seed is not None and str(args.seed).strip().lower() != "none":\n        torch.manual_seed(int(args.seed))\n    device = get_device()\n\n    math_data = load_math_train()\n    if args.checkpoint_path:\n        tokenizer = load_tokenizer_only(which_model="base")\n        model = Qwen3Model(QWEN_CONFIG_06_B)\n        state_dict = torch.load(args.checkpoint_path, map_location="cpu")\n        model.load_state_dict(state_dict)\n        model.to(device)\n    else:\n        model, tokenizer = load_model_and_tokenizer(\n            which_model="base", device=device, use_compile=False\n        )\n\n    trained = train_rlvr_grpo(\n        model=model,\n        tokenizer=tokenizer,\n        math_data=math_data,\n        math500_eval_data=load_math500_test(),\n        device=device,\n        steps=args.steps,\n        num_rollouts=args.num_rollouts,\n        max_new_tokens=args.max_new_tokens,\n        temperature=args.temperature,\n        top_p=args.top_p,\n        eval_max_items=args.eval_on_checkpoint,\n        skip_zero_advantage_updates=args.skip_zero_advantage_updates,\n        show_eta=args.show_eta,\n    )\n\n    if torch.cuda.is_available():\n        max_mem_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)\n        print(f"Max CUDA memory allocated: {max_mem_gb:.2f} GB")\n\n    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)\n    torch.save(trained.state_dict(), CHECKPOINT_DIR/"qwen3-0.6B-rlvr-grpo.pth")\n'


## Clear GPU memory
Run this after a failed run, then rerun the embedded training script cell above and the training cell below. If GPU allocated memory remains high, restart the kernel and run from the top.


In [5]:
# Run before training, especially after an interrupted or failed run.
import gc
import sys
import torch

if "namespace" in globals():
    namespace.clear()
    del namespace
for attribute in ("last_traceback", "last_value", "last_exc"):
    if hasattr(sys, attribute):
        setattr(sys, attribute, None)
# IPython may retain the most recent exception separately.
_shell = get_ipython()
if _shell is not None and hasattr(_shell, "_last_traceback"):
    _shell._last_traceback = None
gc.collect()
torch.cuda.empty_cache()
print(f"GPU allocated: {torch.cuda.memory_allocated() / 2**30:.2f} GiB")
print(f"GPU reserved:  {torch.cuda.memory_reserved() / 2**30:.2f} GiB")


GPU allocated: 0.00 GiB
GPU reserved:  0.00 GiB


## Run training
Downloads pretrained Qwen3 0.6B base weights and datasets as needed, then trains all model weights. Output streams below. Interrupting during the training loop triggers the script's interrupt checkpoint. Keep the kernel alive until completion. Evaluation adds time to the displayed training estimate.

In [6]:
import os

script_path = OUTPUT_DIR / "rlvr_grpo_original_no_kl.py"
script_path.write_text(TRAINING_SOURCE, encoding="utf-8")
arguments = [str(script_path),
             "--num_rollouts", str(NUM_ROLLOUTS),
             "--max_new_tokens", str(MAX_NEW_TOKENS),
             "--eval_on_checkpoint", str(EVAL_EXAMPLES),
             "--show_eta", "--skip-zero-advantage-updates"]
if STEPS is not None:
    arguments += ["--steps", str(STEPS)]

previous_argv = sys.argv[:]
previous_cwd = Path.cwd()
try:
    os.chdir(OUTPUT_DIR)
    sys.argv = arguments
    namespace = {"__name__": "__main__", "__file__": str(script_path)}
    exec(compile(TRAINING_SOURCE, str(script_path), "exec"), namespace)
finally:
    sys.argv = previous_argv
    os.chdir(previous_cwd)


Using NVIDIA CUDA GPU
✓ qwen3/qwen3-0.6B-base.pth already up-to-date
[Step 1/100] loss=0.0000 reward_avg=0.000 tok/sec=36.4 avg_resp_len=229.9 | ETA: 1h 23m 26s
[Step 2/100] loss=-14.1698 reward_avg=0.125 tok/sec=36.1 avg_resp_len=251.0 | ETA: 1h 26m 47s
[Step 3/100] loss=0.0000 reward_avg=0.000 tok/sec=34.7 avg_resp_len=6.6 | ETA: 58m 05s
[Step 4/100] loss=0.6065 reward_avg=0.250 tok/sec=23.1 avg_resp_len=6.6 | ETA: 44m 02s
[Step 5/100] loss=8.5628 reward_avg=0.125 tok/sec=34.3 avg_resp_len=72.0 | ETA: 40m 10s
[Step 6/100] loss=18.3609 reward_avg=0.375 tok/sec=35.3 avg_resp_len=175.6 | ETA: 43m 31s
[Step 7/100] loss=0.0000 reward_avg=0.000 tok/sec=36.2 avg_resp_len=645.5 | ETA: 1h 08m 31s
[Step 8/100] loss=6.9605 reward_avg=0.750 tok/sec=35.5 avg_resp_len=156.1 | ETA: 1h 06m 03s
[Step 9/100] loss=0.0000 reward_avg=0.000 tok/sec=36.1 avg_resp_len=286.8 | ETA: 1h 08m 47s
[Step 10/100] loss=0.0000 reward_avg=0.000 tok/sec=36.2 avg_resp_len=447.0 | ETA: 1h 16m 02s
[Step 11/100] loss=-4.63

## Checkpoints
Saved at steps 50 and 100 for this configuration. These contain model weights; they are not complete optimizer-state snapshots. Copy them to persistent storage or download through Jupyter when using a remote kernel.

In [9]:
checkpoint_dir = OUTPUT_DIR / "checkpoints" / "rlvr_grpo_original_no_kl"
print("Checkpoint folder:", checkpoint_dir)
for path in sorted(checkpoint_dir.glob("*.pth")):
    print(path.name, f"{path.stat().st_size / 2**30:.2f} GiB")


Checkpoint folder: /home/jovyan/grpo_training/checkpoints/rlvr_grpo_original_no_kl
qwen3-0.6B-rlvr-grpo-step00050.pth 1.40 GiB
qwen3-0.6B-rlvr-grpo-step00100.pth 1.40 GiB
qwen3-0.6B-rlvr-grpo.pth 1.40 GiB
